# Modelagem de Dados -- Decisões de Design

Todas as decisões são aplicadas sobre um Motor de Crédito -- propostas, clientes, analistas e regras de negócio explícitas.

**Tópicos:**
- Normalização vs Desnormalização (1NF, 2NF, 3NF)
- JSONB vs Colunas Tipadas
- Chaves: UUID vs SERIAL vs UUID v7
- Soft Delete vs Hard Delete vs Histórico
- Audit Trail: trigger-based vs application-level
- Particionamento de tabelas
- DDL completo do Motor de Crédito com justificativas

**Pré-requisitos:** SQL intermediário, PostgreSQL funcional com psycopg2.

## Diagrama ER — Motor de Crédito


![Diagrama ER do Motor de Crédito](assets/04-modelagem-er.png)

## Setup -- Conexão e Utilitários

Para rodar localmente com Docker (banco `fintech_kb`):
```bash
docker run --name pg-fintech -e POSTGRES_PASSWORD=postgres -e POSTGRES_DB=fintech_kb -p 5432:5432 -d postgres:16
```

In [1]:
import json
import textwrap

# Dependências: pip install psycopg2-binary python-dateutil
# Conexão configurável
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "fintech_kb",
    "user": "postgres",
    "password": "postgres",
}

def get_conn():
    """Retorna conexão psycopg2. Raise se não conseguir conectar."""
    try:
        import psycopg2
        return psycopg2.connect(**DB_CONFIG)
    except ImportError:
        raise ImportError("Instale psycopg2-binary: pip install psycopg2-binary")
    except Exception as e:
        raise ConnectionError(f"Não foi possível conectar ao PostgreSQL: {e}")

def run_ddl(sql: str, label: str = "") -> None:
    """Executa DDL (CREATE, ALTER, DROP) e imprime confirmação."""
    conn = get_conn()
    try:
        with conn:
            with conn.cursor() as cur:
                cur.execute(sql)
        if label:
            print(f"OK: {label}")
    finally:
        conn.close()

def run_query(sql: str, params=None) -> list:
    """Executa SELECT e retorna lista de dicts."""
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            cols = [d[0] for d in cur.description]
            return [dict(zip(cols, row)) for row in cur.fetchall()]
    finally:
        conn.close()

print("Utilitários carregados.")

Utilitários carregados.


## 1. Normalização vs Desnormalização

### 1NF -- Valores Atômicos

Cada coluna armazena um valor único e indivisível. Listas em colunas texto são impossíveis de filtrar com índice e quebram JOINs.

```sql
-- ERRADO: viola 1NF
CREATE TABLE clientes_bad (
    id     bigint PRIMARY KEY,
    nome   varchar(200),
    fones  varchar(100)  -- "(11)9999-0001, (11)9999-0002"
);

-- CORRETO: 1NF
CREATE TABLE clientes (
    id   bigint PRIMARY KEY,
    nome varchar(200)
);

CREATE TABLE clientes_fones (
    id         bigint PRIMARY KEY GENERATED ALWAYS AS IDENTITY,
    cliente_id bigint REFERENCES clientes(id),
    fone       varchar(20) NOT NULL,
    tipo       varchar(10) CHECK (tipo IN ('celular','fixo','comercial'))
);
```

### 2NF -- Sem Dependências Parciais

Em chaves compostas, toda coluna não-chave deve depender da chave inteira. Se `nome_produto` depende apenas de `produto_id` dentro de uma PK `(proposta_id, produto_id)`, é dependência parcial.

```sql
-- ERRADO: nome_produto depende só de produto_id
CREATE TABLE proposta_produto_bad (
    proposta_id  bigint,
    produto_id   bigint,
    nome_produto varchar(100),  -- dependência parcial!
    taxa_juros   numeric(5,2),
    PRIMARY KEY (proposta_id, produto_id)
);

-- CORRETO: 2NF — mover nome_produto para tabela de produtos
CREATE TABLE produtos (
    id           bigint PRIMARY KEY GENERATED ALWAYS AS IDENTITY,
    nome_produto varchar(100) NOT NULL
);

CREATE TABLE proposta_produto (
    proposta_id bigint REFERENCES propostas(id),
    produto_id  bigint REFERENCES produtos(id),
    taxa_juros  numeric(5,2) NOT NULL,
    PRIMARY KEY (proposta_id, produto_id)
);
```

### 3NF -- Sem Dependências Transitivas

Coluna não-chave não pode depender de outra coluna não-chave. O caso clássico: `cep` determina `cidade` e `estado`, que portanto dependem transitivamente da PK.

```sql
-- ERRADO: cep determina cidade e estado (dependência transitiva)
CREATE TABLE enderecos_bad (
    id     bigint PRIMARY KEY,
    cep    char(8),
    cidade varchar(100),  -- depende de cep, não de id
    estado char(2)        -- depende de cep, não de id
);

-- CORRETO: 3NF
CREATE TABLE ceps (
    cep    char(8)      PRIMARY KEY,
    cidade varchar(100) NOT NULL,
    estado char(2)      NOT NULL
);

CREATE TABLE enderecos (
    id  bigint PRIMARY KEY,
    cep char(8) REFERENCES ceps(cep)
);
```

### Quando Desnormalizar

Desnormalização introduz redundância deliberada para eliminar JOINs em caminhos críticos de leitura.

**Casos válidos:**
- **Dashboards de reporting** sobre bilhões de linhas -- o custo de JOIN domina o tempo de query
- **Tabelas de snapshot** -- você quer o dado como era naquele momento, não o atual
- **Colunas calculadas** caras de recalcular on-the-fly (ex: score agregado)

**Exemplo no Motor de Crédito:** desnormalizar `nome_cliente` e `cpf` em `propostas` para relatórios:

```sql
-- Tabela desnormalizada para reporting
CREATE TABLE propostas_report AS
SELECT
    p.id,
    p.uuid,
    c.nome  AS nome_cliente,
    c.cpf   AS cpf_cliente,
    p.valor_solicitado,
    p.status,
    p.criada_em
FROM propostas p
JOIN clientes c ON c.id = p.cliente_id;
```

**Trade-off:** `nome_cliente` em `propostas_report` fica desatualizado até o próximo refresh. Isso é dados desatualizados entre refreshes (staleness de leitura) -- não inconsistência do banco, é uma escolha explícita de performance sobre frescor do dado.

In [2]:
# Demonstração: comparar custo de JOIN vs coluna desnormalizada
# (executa em banco local se disponível; caso contrário, exibe as queries)

query_normalizada = """
EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
SELECT p.id, c.nome, p.valor_solicitado, p.status
FROM propostas p
JOIN clientes c ON c.id = p.cliente_id
WHERE p.status = 'aprovada'
  AND p.criada_em >= NOW() - INTERVAL '30 days';
"""

query_desnormalizada = """
EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
SELECT id, nome_cliente, valor_solicitado, status
FROM propostas_report
WHERE status = 'aprovada'
  AND criada_em >= NOW() - INTERVAL '30 days';
"""

print("Query normalizada (com JOIN):")
print(textwrap.indent(query_normalizada.strip(), "  "))

print("\nQuery desnormalizada (sem JOIN):")
print(textwrap.indent(query_desnormalizada.strip(), "  "))

print("""
Resumo do trade-off:
  Normalizada  : sempre consistente, custo de JOIN em leitura
  Desnormalizada: consistência eventual, leitura mais rápida
""")

Query normalizada (com JOIN):
  EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
  SELECT p.id, c.nome, p.valor_solicitado, p.status
  FROM propostas p
  JOIN clientes c ON c.id = p.cliente_id
  WHERE p.status = 'aprovada'
    AND p.criada_em >= NOW() - INTERVAL '30 days';

Query desnormalizada (sem JOIN):
  EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
  SELECT id, nome_cliente, valor_solicitado, status
  FROM propostas_report
  WHERE status = 'aprovada'
    AND criada_em >= NOW() - INTERVAL '30 days';

Resumo do trade-off:
  Normalizada  : sempre consistente, custo de JOIN em leitura
  Desnormalizada: consistência eventual, leitura mais rápida



## 2. JSONB vs Colunas Tipadas

### Quando usar JSONB

O schema é variável ou controlado por terceiros:

- **Response de APIs externas** (Bureau de crédito, Serasa, Receita Federal): cada provedor retorna campos diferentes
- **Configurações de produto** com campos opcionais por tipo
- **Eventos de audit** onde você preserva o snapshot completo do registro

```sql
-- JSONB para response de bureau: schema varia por provedor
CREATE TABLE bureau_consultas (
    id            bigserial    PRIMARY KEY,
    proposta_id   bigint       NOT NULL REFERENCES propostas(id),
    provedor      varchar(50)  NOT NULL,  -- 'serasa', 'boa_vista', 'spc'
    score_retornado integer,              -- campo comum -> coluna tipada
    response_data jsonb        NOT NULL   -- tudo mais -> JSONB
);
```

### Quando usar Colunas Tipadas

O schema é conhecido, estável e filtrado/ordenado:

- **Campos de negócio** de propostas: `valor_solicitado`, `prazo_meses`, `status`
- **Campos de cliente**: `cpf`, `nome`, `renda_mensal`
- **Qualquer campo que aparece em WHERE, ORDER BY ou JOIN**

### Abordagem Híbrida (recomendada)

```sql
CREATE TABLE propostas (
    id               bigserial    PRIMARY KEY,
    -- campos consultados: colunas tipadas com índices
    status           varchar(30)  NOT NULL,
    score_calculado  numeric(6,2),
    valor_solicitado numeric(15,2) NOT NULL,
    -- campos variados/contextuais: JSONB
    metadata         jsonb        DEFAULT '{}'
);
```

### Performance: Indexação JSONB

JSONB suporta índice GIN para buscas dentro do documento e índices em caminhos específicos:

```sql
-- Índice GIN geral (para operadores @>, ?, ?|, ?&)
CREATE INDEX idx_bureau_response_gin
    ON bureau_consultas USING GIN (response_data);

-- Índice em campo específico do JSONB (mais seletivo)
CREATE INDEX idx_bureau_response_score
    ON bureau_consultas ((response_data->>'score_pf'));

-- Busca usando o índice GIN
SELECT * FROM bureau_consultas
WHERE response_data @> '{"inadimplente": true}';
```

**Custo por operação:** B-Tree em coluna tipada tem custo por lookup menor que GIN para buscas de igualdade simples. GIN compensa quando você precisa buscar por qualquer campo dentro de um documento grande -- a flexibilidade do operador `@>` justifica o overhead por operação.

In [3]:
# Demonstração: diferença de acesso entre coluna tipada e JSONB

exemplos_acesso_jsonb = [
    {
        "descricao": "Acessar campo top-level",
        "sql": "SELECT response_data->>'score_pf' FROM bureau_consultas WHERE id = 1;",
        "nota": "->> retorna TEXT; -> retorna JSONB"
    },
    {
        "descricao": "Acessar campo aninhado",
        "sql": "SELECT response_data->'dividas'->0->>'valor' FROM bureau_consultas WHERE id = 1;",
        "nota": "Cadeia de operadores para path aninhado"
    },
    {
        "descricao": "Verificar se campo existe",
        "sql": "SELECT * FROM bureau_consultas WHERE response_data ? 'score_pf';",
        "nota": "Operador ? verifica existência de chave"
    },
    {
        "descricao": "Filtrar por valor dentro do JSON",
        "sql": "SELECT * FROM bureau_consultas WHERE response_data @> '{\"inadimplente\": true}';",
        "nota": "@> usa índice GIN se existir"
    },
    {
        "descricao": "Atualizar campo específico sem sobrescrever tudo",
        "sql": "UPDATE bureau_consultas SET response_data = response_data || '{\"reprocessado\": true}' WHERE id = 1;",
        "nota": "|| faz merge; jsonb_set para campos aninhados"
    },
]

print("Operadores JSONB principais:\n")
for ex in exemplos_acesso_jsonb:
    print(f"  [{ex['descricao']}]")
    print(f"  SQL  : {ex['sql']}")
    print(f"  Nota : {ex['nota']}")
    print()

print("""
Regra prática:
  - Campo aparece em WHERE/ORDER BY/JOIN -> coluna tipada + índice B-Tree
  - Campo é variável por provedor/contexto -> JSONB
  - Campo JSONB aparece frequentemente em WHERE -> índice GIN ou índice em path específico
""")

Operadores JSONB principais:

  [Acessar campo top-level]
  SQL  : SELECT response_data->>'score_pf' FROM bureau_consultas WHERE id = 1;
  Nota : ->> retorna TEXT; -> retorna JSONB

  [Acessar campo aninhado]
  SQL  : SELECT response_data->'dividas'->0->>'valor' FROM bureau_consultas WHERE id = 1;
  Nota : Cadeia de operadores para path aninhado

  [Verificar se campo existe]
  SQL  : SELECT * FROM bureau_consultas WHERE response_data ? 'score_pf';
  Nota : Operador ? verifica existência de chave

  [Filtrar por valor dentro do JSON]
  SQL  : SELECT * FROM bureau_consultas WHERE response_data @> '{"inadimplente": true}';
  Nota : @> usa índice GIN se existir

  [Atualizar campo específico sem sobrescrever tudo]
  SQL  : UPDATE bureau_consultas SET response_data = response_data || '{"reprocessado": true}' WHERE id = 1;
  Nota : || faz merge; jsonb_set para campos aninhados


Regra prática:
  - Campo aparece em WHERE/ORDER BY/JOIN -> coluna tipada + índice B-Tree
  - Campo é variável por

## 3. Chaves: UUID vs SERIAL vs UUID v7

A escolha do tipo de chave primária afeta performance de inserção, legibilidade de logs e capacidade de integração entre serviços.

### SERIAL / BIGSERIAL

Sequência autoincrementada do PostgreSQL. Valores são sequenciais, o que é ótimo para índices B-Tree (inserções sempre no final da árvore — sem fragmentação).

```sql
-- SERIAL: inteiro de 4 bytes (max ~2 bilhões)
CREATE TABLE analistas (
    id bigserial PRIMARY KEY  -- BIGSERIAL: 8 bytes, max ~9 quintilhões
);

-- Equivalente explícito (preferido no PostgreSQL moderno)
CREATE TABLE analistas (
    id bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY
);
```

**Vantagens:** simples, compacto, índice B-Tree eficiente, legível em logs.
**Limitação:** não funciona em sistemas distribuídos — dois nós podem gerar o mesmo ID.

### UUID v4

Identificador universalmente único de 128 bits, gerado com aleatoriedade total.

```sql
-- UUID v4: aleatório, sem ordem temporal
CREATE EXTENSION IF NOT EXISTS "pgcrypto";

CREATE TABLE propostas (
    uuid uuid DEFAULT gen_random_uuid() UNIQUE
);
```

**Vantagens:** sem colisão mesmo em sistemas distribuídos, pode ser gerado no cliente.
**Problema crítico:** aleatoriedade total = inserções em posições aleatórias no B-Tree = fragmentação de índice = performance degradada em tabelas grandes (milhões de linhas).

### UUID v7 (recomendado para sistemas distribuídos)

UUID ordenado por tempo (timestamp nos primeiros 48 bits). Combina unicidade global com localidade de B-Tree.

```sql
-- UUID v7 no PostgreSQL 17+ (nativo)
SELECT uuidv7();  -- disponível no pg17+

-- Para versões anteriores: extensão pg_uuidv7
CREATE EXTENSION IF NOT EXISTS "pg_uuidv7";
SELECT uuid_generate_v7();
```

### Decisão para o Motor de Crédito

| Tabela | Chave | Justificativa |
|---|---|---|
| `clientes` | `bigserial` | Banco único, sem distribuição, muitos JOINs |
| `analistas` | `bigserial` | Tabela pequena, banco único |
| `propostas` | `bigserial` (PK) + `uuid` (UK) | PK para JOINs internos eficientes; UUID exposto para APIs externas |
| `bureau_consultas` | `bigserial` | Tabela interna, sem exposição de ID |
| `propostas_audit` | `bigserial` | Append-only, sem necessidade de UUID |

**Padrão recomendado para microsserviços:** `bigserial` como PK interna + `uuid v7` como identificador externo (exposto em APIs). Internamente, todos os JOINs usam `bigint` — rápido. Externamente, nunca se expõe o SERIAL (que revelaria volume de negócio).

In [4]:
import uuid
import time
import struct

def uuid_v4() -> str:
    """UUID v4: aleatório, sem ordem temporal."""
    return str(uuid.uuid4())

def uuid_v7() -> str:
    """
    UUID v7: time-ordered (RFC draft).
    Primeiros 48 bits = timestamp em milissegundos.
    Bits 49-51 = versão (0b0111 = 7).
    Bits restantes = aleatórios.
    """
    ts_ms = int(time.time() * 1000)
    # 48 bits de timestamp + 4 bits de versão (7) + 12 bits aleatórios
    rand_a = uuid.uuid4().int & 0x0FFF
    rand_b = uuid.uuid4().int & ((1 << 62) - 1)

    high = (ts_ms << 16) | (0x7 << 12) | rand_a
    low = (0b10 << 62) | rand_b

    raw = (high << 64) | low
    hex_str = f"{raw:032x}"
    return f"{hex_str[:8]}-{hex_str[8:12]}-{hex_str[12:16]}-{hex_str[16:20]}-{hex_str[20:]}"

# Gerar exemplos
print("UUID v4 (aleatório):")
for _ in range(5):
    print(f"  {uuid_v4()}")

print("\nUUID v7 (time-ordered):")
for _ in range(5):
    time.sleep(0.002)
    print(f"  {uuid_v7()}")

print("""
Observe: UUIDs v7 são lexicograficamente ordenados por tempo.
Inserções em sequência ficam próximas no B-Tree -> menos page splits -> melhor performance.
""")

UUID v4 (aleatório):
  91befac1-0e56-4fdb-9c70-a2eb285d6f53
  c7801e34-925f-458e-ae4f-ad0f721cef0d
  d196a001-51f1-48e8-aeab-f656a5b90b82
  3a5a5012-6ad9-464b-9c32-b8bb17eb3a82
  688a20aa-af06-428b-9b5a-16586e985940

UUID v7 (time-ordered):
  019d0bdb-0441-76ef-a878-6aa31c822e28
  019d0bdb-0443-7f33-8c29-3607b2a695d9
  019d0bdb-0445-738d-acdf-1717a8de778c
  019d0bdb-0447-74cb-ae6d-0671ebbd7f41
  019d0bdb-0449-7ca8-8d8d-6b8ff715b229

Observe: UUIDs v7 são lexicograficamente ordenados por tempo.
Inserções em sequência ficam próximas no B-Tree -> menos page splits -> melhor performance.



## 4. Soft Delete vs Hard Delete vs Histórico

Em sistemas de crédito, dados de clientes e propostas são retidos por anos -- é requisito regulatório (BACEN, LGPD para dados financeiros). A escolha do mecanismo de retenção afeta queries, índices e complexidade operacional.

### Soft Delete (`deleted_at`)

Marca o registro como deletado sem remover fisicamente. A coluna `deleted_at` fica `NULL` para registros ativos e preenchida com o timestamp da exclusão para registros deletados.

```sql
ALTER TABLE clientes ADD COLUMN deleted_at timestamptz DEFAULT NULL;

-- "Deletar" um cliente
UPDATE clientes SET deleted_at = NOW() WHERE id = 42;

-- Toda query PRECISA filtrar
SELECT * FROM clientes WHERE deleted_at IS NULL;
```

**Vantagem:** simples, reversível, sem perda de dado.
**Problema:** toda query precisa de `WHERE deleted_at IS NULL`. Fácil de esquecer. Polui a tabela principal com registros "mortos". Índices ficam maiores (incluem linhas deletadas).

**Mitigação com partial index:**
```sql
-- Índice só para registros ativos
CREATE INDEX idx_clientes_cpf_ativos
    ON clientes (cpf)
    WHERE deleted_at IS NULL;
```

### Hard Delete

Remove fisicamente a linha. Simples e limpo, mas perde o histórico.

```sql
DELETE FROM clientes WHERE id = 42;
```

**Use quando:** dados de teste em homologação, registros sem valor de negócio, compliance exige esquecimento (LGPD direito ao esquecimento).

### Tabela de Histórico

Move o registro deletado para uma tabela de histórico antes de remover da principal. A tabela principal fica limpa e o histórico fica acessível separadamente.

```sql
CREATE TABLE clientes_historico (
    LIKE clientes INCLUDING ALL,  -- copia estrutura completa
    deletado_em  timestamptz NOT NULL DEFAULT NOW(),
    deletado_por bigint
);

-- Mover para histórico e deletar da principal (atômico)
WITH moved AS (
    DELETE FROM clientes WHERE id = 42 RETURNING *
)
INSERT INTO clientes_historico
SELECT *, NOW(), current_setting('app.user_id', true)::bigint FROM moved;
```

**Vantagem:** tabela principal sempre limpa, histórico disponível quando necessário.
**Custo:** esquema duplicado a manter; precisa de processo de limpeza do histórico eventualmente.

### Resumo de Escolha

| Cenário | Recomendação |
|---|---|
| Dados de negócio com auditoria necessária | Tabela de histórico |
| Entidade com relacionamentos (FK) | Soft delete (não quebra FK) |
| LGPD / direito ao esquecimento | Hard delete (+ registro de que foi deletado, sem o dado) |
| Tabela de cache / temporária | Hard delete |

In [5]:
# Demonstração: padrões de soft delete e histórico

soft_delete_ddl = """
-- Soft delete: coluna deleted_at
ALTER TABLE clientes ADD COLUMN IF NOT EXISTS deleted_at timestamptz DEFAULT NULL;

-- Partial index para performance em queries de registros ativos
CREATE INDEX IF NOT EXISTS idx_clientes_ativos_cpf
    ON clientes (cpf)
    WHERE deleted_at IS NULL;

-- View para esconder registros deletados
CREATE OR REPLACE VIEW clientes_ativos AS
    SELECT * FROM clientes WHERE deleted_at IS NULL;
"""

historico_ddl = """
-- Tabela de histórico para clientes deletados
-- LIKE ... INCLUDING ALL copia colunas, constraints, defaults e índices
CREATE TABLE IF NOT EXISTS clientes_historico (
    LIKE clientes INCLUDING ALL,
    -- campos de controle do histórico
    deletado_em      timestamptz  NOT NULL DEFAULT NOW(),
    deletado_por     bigint,
    motivo           text
);
"""

move_to_historico_sql = """
-- Mover para histórico atomicamente (sem risco de inconsistência)
WITH moved AS (
    DELETE FROM clientes
    WHERE id = %(cliente_id)s
    RETURNING *
)
INSERT INTO clientes_historico
SELECT
    moved.*,
    NOW(),
    %(deletado_por)s,
    %(motivo)s
FROM moved;
"""

print("DDL Soft Delete:")
print(textwrap.indent(soft_delete_ddl.strip(), "  "))

print("\nDDL Tabela de Histórico:")
print(textwrap.indent(historico_ddl.strip(), "  "))

print("\nQuery mover para histórico (atômica com CTE):")
print(textwrap.indent(move_to_historico_sql.strip(), "  "))

DDL Soft Delete:
  -- Soft delete: coluna deleted_at
  ALTER TABLE clientes ADD COLUMN IF NOT EXISTS deleted_at timestamptz DEFAULT NULL;

  -- Partial index para performance em queries de registros ativos
  CREATE INDEX IF NOT EXISTS idx_clientes_ativos_cpf
      ON clientes (cpf)
      WHERE deleted_at IS NULL;

  -- View para esconder registros deletados
  CREATE OR REPLACE VIEW clientes_ativos AS
      SELECT * FROM clientes WHERE deleted_at IS NULL;

DDL Tabela de Histórico:
  -- Tabela de histórico para clientes deletados
  -- LIKE ... INCLUDING ALL copia colunas, constraints, defaults e índices
  CREATE TABLE IF NOT EXISTS clientes_historico (
      LIKE clientes INCLUDING ALL,
      -- campos de controle do histórico
      deletado_em      timestamptz  NOT NULL DEFAULT NOW(),
      deletado_por     bigint,
      motivo           text
  );

Query mover para histórico (atômica com CTE):
  -- Mover para histórico atomicamente (sem risco de inconsistência)
  WITH moved AS (
      D

## 5. Audit Trail

Em sistemas financeiros, registrar toda alteração em dados críticos é requisito legal (BACEN, LGPD) e operacional (investigação de fraude, rollback manual). Há duas abordagens complementares.

### Trigger-based (nível de banco)

O banco captura automaticamente cada INSERT/UPDATE/DELETE e registra na tabela de audit.

**Vantagem:** não pode ser bypassado pela aplicação. Mesmo queries diretas via psql são capturadas.
**Desvantagem:** não tem contexto de aplicação (qual usuário da aplicação fez a alteração -- só sabe o usuário do banco).

```sql
-- Tabela de audit
CREATE TABLE propostas_audit (
    id             bigserial    PRIMARY KEY,
    proposta_id    bigint       NOT NULL,
    operacao       char(1)      NOT NULL CHECK (operacao IN ('I','U','D')),
    status_anterior varchar(30),
    status_novo    varchar(30),
    dados_anteriores jsonb,
    dados_novos      jsonb,
    alterado_em    timestamptz  NOT NULL DEFAULT NOW(),
    db_user        text         NOT NULL DEFAULT current_user
);

-- Função trigger
CREATE OR REPLACE FUNCTION audit_propostas()
RETURNS TRIGGER AS $$
BEGIN
    INSERT INTO propostas_audit (
        proposta_id, operacao,
        status_anterior, status_novo,
        dados_anteriores, dados_novos
    ) VALUES (
        COALESCE(NEW.id, OLD.id),
        LEFT(TG_OP, 1),
        CASE WHEN TG_OP != 'INSERT' THEN OLD.status END,
        CASE WHEN TG_OP != 'DELETE' THEN NEW.status END,
        CASE WHEN TG_OP != 'INSERT' THEN row_to_json(OLD)::jsonb END,
        CASE WHEN TG_OP != 'DELETE' THEN row_to_json(NEW)::jsonb END
    );
    RETURN COALESCE(NEW, OLD);
END;
$$ LANGUAGE plpgsql;

-- Trigger dispara em qualquer mudança na tabela propostas
CREATE TRIGGER tg_propostas_audit
    AFTER INSERT OR UPDATE OR DELETE ON propostas
    FOR EACH ROW EXECUTE FUNCTION audit_propostas();
```

### Application-level (nível de aplicação)

A própria aplicação registra o audit antes ou depois de cada operação.

**Vantagem:** tem contexto rico -- usuário da sessão, IP de origem, motivo da alteração, request_id.
**Risco:** pode ser bypassado (queries diretas não passam pela aplicação).

```python
def aprovar_proposta(proposta_id: int, analista_id: int, motivo: str, conn) -> None:
    with conn:
        with conn.cursor() as cur:
            # Busca estado atual
            cur.execute(
                "SELECT status FROM propostas WHERE id = %s FOR UPDATE",
                (proposta_id,)
            )
            row = cur.fetchone()
            if not row:
                raise ValueError("Proposta nao encontrada")

            status_anterior = row[0]

            # Atualiza status
            cur.execute(
                "UPDATE propostas SET status = 'aprovada', analista_id = %s, motivo_decisao = %s WHERE id = %s",
                (analista_id, motivo, proposta_id)
            )

            # Registra audit com contexto da aplicacao
            cur.execute(
                """
                INSERT INTO propostas_audit
                    (proposta_id, operacao, status_anterior, status_novo, alterado_por, motivo)
                VALUES (%s, 'U', %s, 'aprovada', %s, %s)
                """,
                (proposta_id, status_anterior, analista_id, motivo)
            )
```

### Recomendação

Em sistemas financeiros: **use os dois**. Trigger para garantir que nada escapa; application-level para registrar contexto rico. O trigger é a última linha de defesa.

In [6]:
# Função que simula o application-level audit (sem banco real necessário)
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Optional

@dataclass
class AuditRecord:
    proposta_id: int
    operacao: str
    status_anterior: Optional[str]
    status_novo: Optional[str]
    alterado_por: int
    motivo: str
    alterado_em: datetime = field(default_factory=lambda: datetime.now(timezone.utc))

# Simulação de estado em memória
propostas_estado = {
    1: {"id": 1, "status": "em_analise", "analista_id": None},
    2: {"id": 2, "status": "pendente",   "analista_id": None},
}
audit_log: list[AuditRecord] = []

def aprovar_proposta(proposta_id: int, analista_id: int, motivo: str) -> None:
    """Aprova proposta e registra audit trail."""
    proposta = propostas_estado.get(proposta_id)
    if not proposta:
        raise ValueError(f"Proposta {proposta_id} não encontrada")

    status_anterior = proposta["status"]

    # Valida transição de status
    if status_anterior not in ("em_analise", "pendente"):
        raise ValueError(f"Status '{status_anterior}' não pode ser aprovado")

    # Aplica mudança
    proposta["status"] = "aprovada"
    proposta["analista_id"] = analista_id

    # Registra audit com contexto da aplicação
    audit_log.append(AuditRecord(
        proposta_id=proposta_id,
        operacao="U",
        status_anterior=status_anterior,
        status_novo="aprovada",
        alterado_por=analista_id,
        motivo=motivo,
    ))

# Executar operações
aprovar_proposta(proposta_id=1, analista_id=10, motivo="Score acima do mínimo e renda suficiente")
aprovar_proposta(proposta_id=2, analista_id=10, motivo="Cliente sem restrições no bureau")

print("Estado atual das propostas:")
for p in propostas_estado.values():
    print(f"  Proposta {p['id']}: status={p['status']}, analista={p['analista_id']}")

print("\nAudit trail:")
for r in audit_log:
    print(f"  [{r.alterado_em.strftime('%H:%M:%S')}] proposta={r.proposta_id} "
          f"{r.status_anterior} -> {r.status_novo} por analista={r.alterado_por}")
    print(f"    Motivo: {r.motivo}")

Estado atual das propostas:
  Proposta 1: status=aprovada, analista=10
  Proposta 2: status=aprovada, analista=10

Audit trail:
  [15:26:36] proposta=1 em_analise -> aprovada por analista=10
    Motivo: Score acima do mínimo e renda suficiente
  [15:26:36] proposta=2 pendente -> aprovada por analista=10
    Motivo: Cliente sem restrições no bureau


## 6. Particionamento de Tabelas

A partir de ~50 milhões de linhas, queries que filtram por uma coluna específica (data, status, região) se beneficiam de particionamento. O PostgreSQL usa **partition pruning** para ignorar partições inteiras que não satisfazem o filtro -- eliminando I/O antes de começar o scan.

### Range Partition por Data

Ideal para tabelas com crescimento temporal: logs, transações, propostas.

```sql
-- Tabela principal declarada como particionada
CREATE TABLE propostas_particoes (
    id             bigint       NOT NULL,
    criada_em      timestamptz  NOT NULL,
    status         varchar(30)  NOT NULL,
    valor_solicitado numeric(15,2)
) PARTITION BY RANGE (criada_em);

-- Partições mensais criadas com antecedência
CREATE TABLE propostas_2025_01 PARTITION OF propostas_particoes
    FOR VALUES FROM ('2025-01-01') TO ('2025-02-01');

CREATE TABLE propostas_2025_02 PARTITION OF propostas_particoes
    FOR VALUES FROM ('2025-02-01') TO ('2025-03-01');

CREATE TABLE propostas_2025_12 PARTITION OF propostas_particoes
    FOR VALUES FROM ('2025-12-01') TO ('2026-01-01');

-- Partição padrão para valores fora do range (previne erro de inserção)
CREATE TABLE propostas_outros PARTITION OF propostas_particoes DEFAULT;
```

### List Partition por Status

Separa dados quentes (pendente, em_analise) de dados frios (aprovada, rejeitada, expirada).

```sql
CREATE TABLE propostas_por_status (
    id    bigint      NOT NULL,
    status varchar(30) NOT NULL,
    criada_em timestamptz NOT NULL
) PARTITION BY LIST (status);

-- Dados quentes: acessados frequentemente, índices menores
CREATE TABLE propostas_ativas PARTITION OF propostas_por_status
    FOR VALUES IN ('pendente', 'em_analise', 'aguardando_documentos');

-- Dados frios: raramente acessados, podem ser arquivados
CREATE TABLE propostas_finalizadas PARTITION OF propostas_por_status
    FOR VALUES IN ('aprovada', 'rejeitada', 'expirada', 'cancelada');
```

### Partition Pruning

```sql
-- Query que usa partition pruning:
-- PostgreSQL consulta APENAS propostas_2025_01, ignorando as demais
EXPLAIN SELECT * FROM propostas_particoes
WHERE criada_em >= '2025-01-01' AND criada_em < '2025-02-01';

-- Verificar no EXPLAIN: "Partitions: propostas_2025_01"
-- Se aparecer todas as partições, o pruning não está funcionando
-- (geralmente por cast implícito ou função aplicada na coluna)
```

**Cuidado com o filtro:** `WHERE date_trunc('month', criada_em) = '2025-01-01'` **não** usa partition pruning -- o PostgreSQL não consegue inferir o range quando uma função envelopa a coluna de partição. Use sempre comparações diretas.

In [7]:
from datetime import date
from dateutil.relativedelta import relativedelta  # pip install python-dateutil

def generate_monthly_partition_ddl(
    base_table: str,
    start_year: int,
    start_month: int,
    num_months: int,
) -> list[str]:
    """
    Gera DDL para criar partições mensais de uma tabela particionada por range de data.

    Args:
        base_table: nome da tabela pai (ex: 'propostas_particoes')
        start_year: ano inicial
        start_month: mês inicial (1-12)
        num_months: quantidade de partições a gerar

    Returns:
        Lista de statements DDL prontos para execução.
    """
    statements = []
    current = date(start_year, start_month, 1)

    for _ in range(num_months):
        next_month = current + relativedelta(months=1)
        partition_name = f"{base_table}_{current.strftime('%Y_%m')}"

        ddl = (
            f"CREATE TABLE IF NOT EXISTS {partition_name} "
            f"PARTITION OF {base_table} "
            f"FOR VALUES FROM ('{current}') TO ('{next_month}');"
        )
        statements.append(ddl)
        current = next_month

    return statements


# Gerar partições para 2025 e 2026
particoes_ddl = generate_monthly_partition_ddl(
    base_table="propostas_particoes",
    start_year=2025,
    start_month=1,
    num_months=24,
)

print(f"DDL gerado para {len(particoes_ddl)} partições:")
for stmt in particoes_ddl:
    print(f"  {stmt}")

DDL gerado para 24 partições:
  CREATE TABLE IF NOT EXISTS propostas_particoes_2025_01 PARTITION OF propostas_particoes FOR VALUES FROM ('2025-01-01') TO ('2025-02-01');
  CREATE TABLE IF NOT EXISTS propostas_particoes_2025_02 PARTITION OF propostas_particoes FOR VALUES FROM ('2025-02-01') TO ('2025-03-01');
  CREATE TABLE IF NOT EXISTS propostas_particoes_2025_03 PARTITION OF propostas_particoes FOR VALUES FROM ('2025-03-01') TO ('2025-04-01');
  CREATE TABLE IF NOT EXISTS propostas_particoes_2025_04 PARTITION OF propostas_particoes FOR VALUES FROM ('2025-04-01') TO ('2025-05-01');
  CREATE TABLE IF NOT EXISTS propostas_particoes_2025_05 PARTITION OF propostas_particoes FOR VALUES FROM ('2025-05-01') TO ('2025-06-01');
  CREATE TABLE IF NOT EXISTS propostas_particoes_2025_06 PARTITION OF propostas_particoes FOR VALUES FROM ('2025-06-01') TO ('2025-07-01');
  CREATE TABLE IF NOT EXISTS propostas_particoes_2025_07 PARTITION OF propostas_particoes FOR VALUES FROM ('2025-07-01') TO ('2025

## 7. Caso Real — DDL Completo do Motor de Crédito

O design a seguir aplica todas as decisões discutidas. Cada escolha é justificada nos comentários inline.

**Decisões consolidadas:**
- `clientes` e `analistas`: `bigserial` (banco único, muitos JOINs)
- `propostas`: `bigserial` (PK interna) + `uuid` (identificador externo)
- Soft delete em `clientes` (preservar histórico, FK não quebra)
- JSONB em `bureau_consultas.response_data` (schema variável por provedor)
- Audit trigger em `propostas`
- Índices seletivos com partial index para dados ativos
- CHECK constraints em todos os enums
- NOT NULL explícito em campos obrigatórios

In [8]:
DDL_MOTOR_CREDITO = """
-- ============================================================
-- Motor de Crédito -- DDL Completo
-- Decisões de design documentadas inline
-- ============================================================

-- Extensão para UUID (disponível por padrão no PostgreSQL 13+)
CREATE EXTENSION IF NOT EXISTS pgcrypto;


-- ============================================================
-- CLIENTES
-- Decisão: bigserial (banco único, muitos JOINs)
-- Soft delete com deleted_at (preserva histórico, FK intacta)
-- ============================================================
CREATE TABLE clientes (
    id              bigint       PRIMARY KEY GENERATED ALWAYS AS IDENTITY,
    cpf             char(11)     NOT NULL,
    nome            varchar(200) NOT NULL,
    data_nascimento date         NOT NULL,
    email           varchar(254) NOT NULL,
    renda_mensal    numeric(15,2) NOT NULL CHECK (renda_mensal >= 0),
    status          varchar(20)  NOT NULL DEFAULT 'ativo'
                    CHECK (status IN ('ativo', 'bloqueado', 'suspenso')),
    created_at      timestamptz  NOT NULL DEFAULT NOW(),
    updated_at      timestamptz  NOT NULL DEFAULT NOW(),
    deleted_at      timestamptz  DEFAULT NULL  -- soft delete
);

-- Índice único em CPF apenas para registros ativos
CREATE UNIQUE INDEX uq_clientes_cpf_ativos
    ON clientes (cpf)
    WHERE deleted_at IS NULL;

CREATE UNIQUE INDEX uq_clientes_email_ativos
    ON clientes (email)
    WHERE deleted_at IS NULL;


-- ============================================================
-- ANALISTAS
-- Decisão: bigserial (tabela pequena, banco único)
-- Hard delete não usamos; analistas são desativados (ativo=false)
-- ============================================================
CREATE TABLE analistas (
    id        bigint      PRIMARY KEY GENERATED ALWAYS AS IDENTITY,
    matricula varchar(20) NOT NULL UNIQUE,
    nome      varchar(200) NOT NULL,
    email     varchar(254) NOT NULL UNIQUE,
    nivel     varchar(20)  NOT NULL DEFAULT 'junior'
              CHECK (nivel IN ('junior', 'pleno', 'senior', 'gerente')),
    ativo     boolean      NOT NULL DEFAULT true,
    created_at timestamptz NOT NULL DEFAULT NOW()
);


-- ============================================================
-- PROPOSTAS
-- Decisão: bigserial (PK interna) + uuid gerado pelo banco (ID externo)
-- UUID não expõe volume de negócio em APIs externas
-- metadata JSONB para campos variados por tipo de crédito
-- Soft delete com deleted_at (preserva histórico, FK intacta)
-- ============================================================
CREATE TABLE propostas (
    id               bigint       PRIMARY KEY GENERATED ALWAYS AS IDENTITY,
    uuid             uuid         NOT NULL UNIQUE DEFAULT gen_random_uuid(),
    cliente_id       bigint       NOT NULL REFERENCES clientes(id),
    valor_solicitado numeric(15,2) NOT NULL CHECK (valor_solicitado > 0),
    prazo_meses      integer      NOT NULL CHECK (prazo_meses BETWEEN 1 AND 360),
    finalidade       varchar(50)  NOT NULL
                     CHECK (finalidade IN (
                         'pessoal', 'veiculo', 'imovel', 'capital_giro', 'refinanciamento'
                     )),
    status           varchar(30)  NOT NULL DEFAULT 'pendente'
                     CHECK (status IN (
                         'pendente', 'em_analise', 'aguardando_documentos',
                         'aprovada', 'rejeitada', 'expirada', 'cancelada'
                     )),
    score_calculado  numeric(6,2) CHECK (score_calculado BETWEEN 0 AND 1000),
    analista_id      bigint       REFERENCES analistas(id),
    motivo_decisao   text,
    metadata         jsonb        NOT NULL DEFAULT '{}'::jsonb,
    criada_em        timestamptz  NOT NULL DEFAULT NOW(),
    decidida_em      timestamptz,
    updated_at       timestamptz  NOT NULL DEFAULT NOW(),
    deleted_at       timestamptz  DEFAULT NULL  -- soft delete
);

-- Índice para listagem de propostas por cliente (query mais comum)
CREATE INDEX idx_propostas_cliente_status
    ON propostas (cliente_id, status);

-- Índice para fila de trabalho do analista (propostas ativas apenas)
CREATE INDEX idx_propostas_analista_ativos
    ON propostas (analista_id, criada_em DESC)
    WHERE status IN ('em_analise', 'aguardando_documentos');

-- Índice GIN para buscas dentro de metadata
CREATE INDEX idx_propostas_metadata_gin
    ON propostas USING GIN (metadata);


-- ============================================================
-- BUREAU CONSULTAS
-- Decisão: JSONB para response_data (schema varia por provedor)
-- score_retornado como coluna tipada (campo comum a todos os provedores)
-- ============================================================
CREATE TABLE bureau_consultas (
    id               bigint      PRIMARY KEY GENERATED ALWAYS AS IDENTITY,
    proposta_id      bigint      NOT NULL REFERENCES propostas(id),
    provedor         varchar(50) NOT NULL
                     CHECK (provedor IN ('serasa', 'boa_vista', 'spc', 'receita_federal')),
    cpf_consultado   char(11)    NOT NULL,
    score_retornado  integer     CHECK (score_retornado BETWEEN 0 AND 1000),
    response_data    jsonb       NOT NULL,
    status_http      integer     NOT NULL,
    custo_centavos   integer     NOT NULL DEFAULT 0 CHECK (custo_centavos >= 0),
    consultado_em    timestamptz NOT NULL DEFAULT NOW()
);

CREATE INDEX idx_bureau_proposta
    ON bureau_consultas (proposta_id);

CREATE INDEX idx_bureau_response_gin
    ON bureau_consultas USING GIN (response_data);


-- ============================================================
-- PROPOSTAS AUDIT
-- Append-only: nunca atualizado, nunca deletado
-- Captura tanto DB-level (trigger) quanto application-level
-- ============================================================
CREATE TABLE propostas_audit (
    id              bigint      PRIMARY KEY GENERATED ALWAYS AS IDENTITY,
    proposta_id     bigint      NOT NULL,  -- sem FK: registro sobrevive ao delete
    operacao        char(1)     NOT NULL CHECK (operacao IN ('I', 'U', 'D')),
    status_anterior varchar(30),
    status_novo     varchar(30),
    dados_anteriores jsonb,
    dados_novos      jsonb,
    alterado_por    bigint,     -- NULL se alteração direta no banco
    alterado_em     timestamptz NOT NULL DEFAULT NOW(),
    db_user         text        NOT NULL DEFAULT current_user
);

CREATE INDEX idx_audit_proposta_id
    ON propostas_audit (proposta_id, alterado_em DESC);


-- ============================================================
-- TRIGGER DE AUDIT
-- Captura automaticamente qualquer mudança em propostas
-- OLD é NULL em INSERT; NEW é NULL em DELETE
-- ============================================================
CREATE OR REPLACE FUNCTION fn_audit_propostas()
RETURNS TRIGGER AS $$
BEGIN
    INSERT INTO propostas_audit (
        proposta_id,
        operacao,
        status_anterior,
        status_novo,
        dados_anteriores,
        dados_novos
    ) VALUES (
        COALESCE(NEW.id, OLD.id),
        LEFT(TG_OP, 1),
        CASE WHEN TG_OP != 'INSERT' THEN OLD.status END,
        CASE WHEN TG_OP != 'DELETE' THEN NEW.status END,
        CASE WHEN TG_OP != 'INSERT' THEN row_to_json(OLD)::jsonb END,
        CASE WHEN TG_OP != 'DELETE' THEN row_to_json(NEW)::jsonb END
    );
    RETURN COALESCE(NEW, OLD);
END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER tg_propostas_audit
    AFTER INSERT OR UPDATE OR DELETE ON propostas
    FOR EACH ROW EXECUTE FUNCTION fn_audit_propostas();
"""

print("DDL do Motor de Crédito:")
print(DDL_MOTOR_CREDITO)

DDL do Motor de Crédito:

-- ============================================================
-- Motor de Crédito -- DDL Completo
-- Decisões de design documentadas inline
-- ============================================================

-- Extensão para UUID (disponível por padrão no PostgreSQL 13+)
CREATE EXTENSION IF NOT EXISTS pgcrypto;


-- ============================================================
-- CLIENTES
-- Decisão: bigserial (banco único, muitos JOINs)
-- Soft delete com deleted_at (preserva histórico, FK intacta)
-- ============================================================
CREATE TABLE clientes (
    id              bigint       PRIMARY KEY GENERATED ALWAYS AS IDENTITY,
    cpf             char(11)     NOT NULL,
    nome            varchar(200) NOT NULL,
    data_nascimento date         NOT NULL,
    email           varchar(254) NOT NULL,
    renda_mensal    numeric(15,2) NOT NULL CHECK (renda_mensal >= 0),
    status          varchar(20)  NOT NULL DEFAULT 'ativo'
        

In [9]:
# Para executar o DDL em banco real (requer PostgreSQL local configurado)
def executar_ddl_motor_credito(ddl: str) -> None:
    """
    Executa o DDL completo do Motor de Crédito.
    Requer banco configurado conforme DSN no início do notebook.
    """
    conn = get_conn()
    try:
        with conn:
            with conn.cursor() as cur:
                cur.execute(ddl)
        print("DDL executado com sucesso.")
    except Exception as e:
        print(f"Erro ao executar DDL: {e}")
    finally:
        conn.close()


# Verificar índices criados
query_indices = """
SELECT
    t.relname  AS tabela,
    i.relname  AS indice,
    ix.indisunique AS unico,
    am.amname  AS tipo
FROM pg_class t
JOIN pg_index ix  ON t.oid = ix.indrelid
JOIN pg_class i   ON i.oid = ix.indexrelid
JOIN pg_am am     ON i.relam = am.oid
WHERE t.relname IN (
    'clientes', 'analistas', 'propostas', 'bureau_consultas', 'propostas_audit'
)
ORDER BY t.relname, i.relname;
"""

print("Para executar no banco:")
print("  executar_ddl_motor_credito(DDL_MOTOR_CREDITO)")
print()
print("Para verificar índices criados:")
print(textwrap.indent(query_indices.strip(), "  "))

Para executar no banco:
  executar_ddl_motor_credito(DDL_MOTOR_CREDITO)

Para verificar índices criados:
  SELECT
      t.relname  AS tabela,
      i.relname  AS indice,
      ix.indisunique AS unico,
      am.amname  AS tipo
  FROM pg_class t
  JOIN pg_index ix  ON t.oid = ix.indrelid
  JOIN pg_class i   ON i.oid = ix.indexrelid
  JOIN pg_am am     ON i.relam = am.oid
  WHERE t.relname IN (
      'clientes', 'analistas', 'propostas', 'bureau_consultas', 'propostas_audit'
  )
  ORDER BY t.relname, i.relname;


## Resumo -- Gotchas e Dicas Práticas

### Normalização
- 3NF elimina anomalias de atualização. Desnormalize apenas para reporting ou snapshots -- e documente o motivo.
- Tabelas desnormalizadas ficam stale entre refreshes. Não confunda isso com "consistência eventual" (que é um conceito de sistemas distribuídos).

### JSONB
- B-Tree em coluna tipada tem custo por lookup menor que GIN. Só use JSONB para schemas variáveis (APIs externas, configurações dinâmicas).
- `response_data @> '{"key": "val"}'` usa índice GIN; `response_data->>'key' = 'val'` precisa de índice funcional separado.
- `jsonb_set` atualiza campo aninhado sem sobrescrever o documento inteiro. `||` faz merge top-level.

### Chaves
- Nunca exponha SERIAL em APIs públicas -- revela volume de negócio. Use UUID como identificador externo.
- UUID v4 fragmenta B-Tree em tabelas grandes (>10M linhas). UUID v7 resolve isso com prefixo temporal, mas só é nativo no PostgreSQL 17+.

### Delete
- Soft delete exige `WHERE deleted_at IS NULL` em toda query. Esqueceu? Bug silencioso que retorna registros "mortos".
- Partial index com `WHERE deleted_at IS NULL` mantém o índice enxuto -- sem ele, o índice inclui todas as linhas deletadas.
- Para LGPD (direito ao esquecimento), hard delete é obrigatório, mas registre *que* foi deletado (sem o dado pessoal).

### Audit Trail
- Trigger de audit com `OLD.status` quebra em INSERT (OLD é NULL). Sempre use `CASE WHEN TG_OP != 'INSERT'`.
- `RETURN NEW` no trigger também quebra em DELETE. Use `RETURN COALESCE(NEW, OLD)`.
- Trigger-based e application-level não são alternativas -- use os dois em sistemas financeiros.

### Particionamento
- Threshold prático: ~50M linhas com filtro sistemático em uma coluna.
- `WHERE date_trunc('month', col)` mata partition pruning. Sempre filtre com comparações diretas na coluna de partição.
- Partição DEFAULT é obrigatória -- sem ela, inserções fora do range falham com erro.

## Exercícios

### Exercício 1 -- Normalização

A tabela abaixo viola 3NF. Identifique a dependência transitiva e proponha o DDL normalizado:

```sql
CREATE TABLE propostas_denorm (
    id              bigint PRIMARY KEY,
    cliente_cpf     char(11),
    cliente_nome    varchar(200),   -- depende de cliente_cpf
    cliente_renda   numeric(15,2),  -- depende de cliente_cpf
    valor_solicitado numeric(15,2),
    status          varchar(30)
);
```

**Questão:** Qual a dependência transitiva? Escreva o DDL normalizado em 3NF.

---

### Exercício 2 -- JSONB ou Coluna

Para cada campo abaixo, decida: `coluna tipada` ou `JSONB`. Justifique.

1. `taxa_juros_negociada` -- campo de negócio, filtrado em relatórios
2. `dados_veiculo` -- financiamento de veículo: marca, modelo, ano, placa, renavam (só existe para finalidade='veiculo')
3. `webhook_payload` -- corpo bruto do webhook recebido de parceiro externo
4. `score_final` -- resultado do cálculo interno de score, consultado em dashboard

---

### Exercício 3 -- Soft Delete com Partial Index

Dado o seguinte cenário:
- `propostas` usa soft delete com `deleted_at`
- A query mais frequente é: propostas de um cliente ativas, ordenadas por data
- Há 50 milhões de propostas, sendo 2% ativas

**Questão:** Escreva o DDL do índice ideal para essa query. Por que um índice parcial é melhor que um índice normal aqui?

---

### Exercício 4 -- DDL Completo

Desenhe o DDL para um módulo de `documentos` do Motor de Crédito com os requisitos:

1. Cada proposta pode ter múltiplos documentos
2. Tipos de documento: `comprovante_renda`, `documento_identidade`, `comprovante_residencia`, `contrato`
3. Documentos têm status: `pendente`, `aprovado`, `rejeitado`
4. Precisa de audit trail das mudanças de status
5. Documentos nunca são hard deleted (compliance)
6. O caminho do arquivo fica em storage externo (S3) -- guardar apenas a URL
7. Documentos rejeitados precisam ter motivo obrigatório

Inclua: DDL das tabelas, índices, constraints e trigger de audit.

---

### Exercício 5 -- Particionamento

A tabela `bureau_consultas` tem 500 milhões de linhas e cresce 2 milhões por dia. Queries típicas filtram por `consultado_em` nos últimos 90 dias.

**Questão:** Projete a estratégia de particionamento. Escreva:
1. DDL da tabela pai particionada
2. DDL de duas partições exemplo
3. Como garantir que partições antigas sejam arquivadas automaticamente (hint: `pg_partman`)
4. Por que `WHERE date_trunc('month', consultado_em) = '2025-01-01'` não usa partition pruning?

## Respostas dos Exercícios

### Resposta 1 -- Normalização

**Dependência transitiva:** `cliente_nome` e `cliente_renda` dependem de `cliente_cpf`, que não é a PK. A cadeia é: `id -> cliente_cpf -> {cliente_nome, cliente_renda}`. Isso viola 3NF porque colunas não-chave dependem de outra coluna não-chave.

```sql
-- 3NF: separar clientes em tabela própria
CREATE TABLE clientes (
    cpf          char(11)     PRIMARY KEY,
    nome         varchar(200) NOT NULL,
    renda_mensal numeric(15,2) NOT NULL CHECK (renda_mensal >= 0)
);

CREATE TABLE propostas (
    id               bigint       PRIMARY KEY,
    cliente_cpf      char(11)     NOT NULL REFERENCES clientes(cpf),
    valor_solicitado numeric(15,2) NOT NULL,
    status           varchar(30)  NOT NULL
);
```

A alternativa com `bigserial` como PK de clientes (e cpf como UK) é ainda melhor para JOINs -- `bigint` ocupa 8 bytes vs 11 bytes do `char(11)` como FK.

### Resposta 2 -- JSONB ou Coluna

1. **`taxa_juros_negociada`** -- **Coluna tipada** (`numeric(5,4)`). Campo de negócio estável, filtrado e ordenado em relatórios. Precisa de índice B-Tree e validação via CHECK constraint.

2. **`dados_veiculo`** -- **JSONB**. Existe apenas para `finalidade='veiculo'`, ou seja, a maioria das linhas não tem esse dado. Colocar como colunas separadas (`marca`, `modelo`, `ano`, `placa`, `renavam`) criaria 5 colunas NULL na maioria das linhas. JSONB com `CHECK (finalidade != 'veiculo' OR dados_veiculo IS NOT NULL)` resolve.

3. **`webhook_payload`** -- **JSONB**. Schema controlado por terceiro, pode mudar sem aviso. Guardar o payload bruto para debug e reprocessamento. Não indexar (raro filtrar por conteúdo de webhook).

4. **`score_final`** -- **Coluna tipada** (`numeric(6,2)` ou `integer`). Consultado em dashboard, aparece em WHERE e ORDER BY. Precisa de índice B-Tree para performance.

### Resposta 3 -- Soft Delete com Partial Index

```sql
-- Índice parcial: só registros ativos (2% da tabela = ~1M linhas)
CREATE INDEX idx_propostas_cliente_ativas
    ON propostas (cliente_id, criada_em DESC)
    WHERE deleted_at IS NULL;
```

**Por que partial index?**

Com 50M de linhas e apenas 2% ativas (1M), um índice normal indexaria todas as 50M linhas. O partial index indexa apenas 1M -- ocupa ~50x menos espaço em disco e em memória (shared_buffers).

Na prática:
- Índice normal: ~50M entradas, ~1.2 GB
- Partial index: ~1M entradas, ~25 MB

O partial index cabe inteiro no cache do PostgreSQL, enquanto o normal provavelmente não. Isso significa menos I/O de disco em toda query que filtra registros ativos.

A query correspondente:
```sql
SELECT * FROM propostas
WHERE cliente_id = 42
  AND deleted_at IS NULL
ORDER BY criada_em DESC;
```

O PostgreSQL reconhece que `deleted_at IS NULL` na query casa com o predicado do partial index e usa o índice.

### Resposta 4 -- DDL Completo (Módulo Documentos)

```sql
-- ============================================================
-- DOCUMENTOS
-- Soft delete (compliance); audit trail via trigger
-- ============================================================
CREATE TABLE documentos (
    id           bigint       PRIMARY KEY GENERATED ALWAYS AS IDENTITY,
    proposta_id  bigint       NOT NULL REFERENCES propostas(id),
    tipo         varchar(30)  NOT NULL
                 CHECK (tipo IN (
                     'comprovante_renda', 'documento_identidade',
                     'comprovante_residencia', 'contrato'
                 )),
    status       varchar(20)  NOT NULL DEFAULT 'pendente'
                 CHECK (status IN ('pendente', 'aprovado', 'rejeitado')),
    url_storage  text         NOT NULL,  -- S3 URL
    motivo_rejeicao text,
    uploaded_por bigint,
    revisado_por bigint       REFERENCES analistas(id),
    created_at   timestamptz  NOT NULL DEFAULT NOW(),
    updated_at   timestamptz  NOT NULL DEFAULT NOW(),
    deleted_at   timestamptz  DEFAULT NULL,  -- soft delete (nunca hard delete)

    -- Motivo obrigatório quando rejeitado
    CONSTRAINT chk_motivo_rejeicao CHECK (
        status != 'rejeitado' OR motivo_rejeicao IS NOT NULL
    )
);

-- Documentos de uma proposta (query mais comum)
CREATE INDEX idx_documentos_proposta
    ON documentos (proposta_id)
    WHERE deleted_at IS NULL;

-- Documentos pendentes de revisão
CREATE INDEX idx_documentos_pendentes
    ON documentos (created_at)
    WHERE status = 'pendente' AND deleted_at IS NULL;


-- ============================================================
-- AUDIT TRAIL DE DOCUMENTOS
-- ============================================================
CREATE TABLE documentos_audit (
    id              bigint      PRIMARY KEY GENERATED ALWAYS AS IDENTITY,
    documento_id    bigint      NOT NULL,
    operacao        char(1)     NOT NULL CHECK (operacao IN ('I', 'U', 'D')),
    status_anterior varchar(20),
    status_novo     varchar(20),
    dados_anteriores jsonb,
    dados_novos      jsonb,
    alterado_em     timestamptz NOT NULL DEFAULT NOW(),
    db_user         text        NOT NULL DEFAULT current_user
);

CREATE INDEX idx_audit_documento_id
    ON documentos_audit (documento_id, alterado_em DESC);


-- ============================================================
-- TRIGGER DE AUDIT
-- ============================================================
CREATE OR REPLACE FUNCTION fn_audit_documentos()
RETURNS TRIGGER AS $$
BEGIN
    INSERT INTO documentos_audit (
        documento_id, operacao,
        status_anterior, status_novo,
        dados_anteriores, dados_novos
    ) VALUES (
        COALESCE(NEW.id, OLD.id),
        LEFT(TG_OP, 1),
        CASE WHEN TG_OP != 'INSERT' THEN OLD.status END,
        CASE WHEN TG_OP != 'DELETE' THEN NEW.status END,
        CASE WHEN TG_OP != 'INSERT' THEN row_to_json(OLD)::jsonb END,
        CASE WHEN TG_OP != 'DELETE' THEN row_to_json(NEW)::jsonb END
    );
    RETURN COALESCE(NEW, OLD);
END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER tg_documentos_audit
    AFTER INSERT OR UPDATE OR DELETE ON documentos
    FOR EACH ROW EXECUTE FUNCTION fn_audit_documentos();
```

**Decisões:**
- `CONSTRAINT chk_motivo_rejeicao` garante que documentos rejeitados sem motivo falham no banco -- não depende da aplicação validar.
- Partial index em `status = 'pendente'` acelera a fila de revisão sem custo nos documentos já finalizados.
- Sem FK no audit (`documento_id`): o registro de audit sobrevive mesmo se o documento for deletado no futuro.

### Resposta 5 -- Particionamento

**1. DDL da tabela pai particionada:**

```sql
CREATE TABLE bureau_consultas_part (
    id               bigint       NOT NULL GENERATED ALWAYS AS IDENTITY,
    proposta_id      bigint       NOT NULL,
    provedor         varchar(50)  NOT NULL,
    cpf_consultado   char(11)     NOT NULL,
    score_retornado  integer,
    response_data    jsonb        NOT NULL,
    status_http      integer      NOT NULL,
    custo_centavos   integer      NOT NULL DEFAULT 0,
    consultado_em    timestamptz  NOT NULL DEFAULT NOW()
) PARTITION BY RANGE (consultado_em);
```

**2. Duas partições exemplo:**

```sql
CREATE TABLE bureau_consultas_2025_01 PARTITION OF bureau_consultas_part
    FOR VALUES FROM ('2025-01-01') TO ('2025-02-01');

CREATE TABLE bureau_consultas_2025_02 PARTITION OF bureau_consultas_part
    FOR VALUES FROM ('2025-02-01') TO ('2025-03-01');

-- Partição default para evitar erro em inserções fora do range
CREATE TABLE bureau_consultas_default PARTITION OF bureau_consultas_part DEFAULT;
```

**3. Arquivamento automático com pg_partman:**

```sql
-- Instalar extensão
CREATE EXTENSION IF NOT EXISTS pg_partman;

-- Configurar gerenciamento automático
SELECT partman.create_parent(
    p_parent_table := 'public.bureau_consultas_part',
    p_control := 'consultado_em',
    p_type := 'range',
    p_interval := '1 month',
    p_premake := 3  -- criar 3 partições futuras com antecedência
);

-- Configurar retenção: dropar partições com mais de 12 meses
UPDATE partman.part_config
SET retention = '12 months',
    retention_keep_table = false
WHERE parent_table = 'public.bureau_consultas_part';

-- pg_partman precisa de job agendado (pg_cron ou cron do SO)
-- para executar partman.run_maintenance() periodicamente
```

**4. Por que `date_trunc` mata partition pruning:**

```sql
-- NÃO usa pruning:
WHERE date_trunc('month', consultado_em) = '2025-01-01'
```

O PostgreSQL faz partition pruning comparando o predicado da query diretamente contra os bounds das partições. Quando uma função envelopa a coluna (`date_trunc`), o planner não consegue inverter a função para deduzir o range original -- ele não sabe que `date_trunc('month', x) = '2025-01-01'` implica `x >= '2025-01-01' AND x < '2025-02-01'`. O resultado: scan de todas as partições.

```sql
-- USA pruning (correto):
WHERE consultado_em >= '2025-01-01' AND consultado_em < '2025-02-01'
```

Com 500M de linhas em partições mensais (~60M/mês), a diferença entre scan de 1 partição vs todas é de 1-2 ordens de magnitude em tempo de execução.